In [15]:
import json
import base64
import pandas as pd
import time
from pathlib import Path
from PIL import Image
from openai import OpenAI

In [14]:
# 1. Define your local path to the downloaded ChartQA folder
dataset_path = "/Users/smeh/Desktop/Thesis/DChartQA/ChartQA Dataset" 

def load_chart_qa_split(split_name, type="human"):
    """
    Loads ChartQA data for a specific split (train, val, test).
    Type can be 'human' (human-authored) or 'augmented' (machine-generated).
    """
    # Define file paths based on GitHub repository structure
    json_file = os.path.join(dataset_path, split_name, f"{split_name}_{type}.json")
    image_dir = os.path.join(dataset_path, split_name, "png")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Convert to a DataFrame for easier manipulation
    df = pd.DataFrame(data)
    
    # Add absolute image paths to the DataFrame
    df['image_path'] = df['imgname'].apply(lambda x: os.path.join(image_dir, x))
    
    return df

# Example Usage: Load the human-authored training split
train_df = load_chart_qa_split("train", type="human")

# Access the first entry
first_entry = train_df.iloc[0]
print(f"Question: {first_entry['query']}")
print(f"Answer: {first_entry['label']}")

# To display the chart image
# img = Image.open(first_entry['image_path'])
# img.show()


Question: Is the value of Favorable 38 in 2015?
Answer: Yes


In [21]:
# 1. Configuration
dataset_path = Path("/Users/smeh/Desktop/Thesis/DChartQA/ChartQA Dataset")
NGROK_URL = "https://606c61eefd92.ngrok-free.app/v1"

# List of models to benchmark
# NOTE: Ensure these exact names exist in 'ollama list' on your remote PC
MODELS_TO_TEST = ["llava:7b", "qwen2.5vl:7b", "qwen3-vl:8b"]

client = OpenAI(
    base_url=NGROK_URL,
    api_key="ollama",
    timeout=180.0 
)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_chart_response(model_name, query, image_path):
    base64_image = encode_image(image_path)
    
    # System prompt for concise answers
    system_msg = "You are a helpful AI assistant capable of reading charts and graphs. Answer the user's question concisely based on the image provided. Give me very short answers."
    
    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": query},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    },
                ],
            }
        ],
        max_tokens=500,
        temperature=0.1
    )
    
    end_time = time.time()
    duration = end_time - start_time
    
    return response.choices[0].message.content, duration

def load_chart_qa_split(split_name, type="human"):
    json_file = dataset_path / split_name / f"{split_name}_{type}.json"
    image_dir = dataset_path / split_name / "png"
    
    if not json_file.exists():
        print(f"ERROR: Could not find file at {json_file}")
        return None
    
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    df['image_path'] = df['imgname'].apply(lambda x: str(image_dir / x))
    
    return df

def check_correctness(model_ans, label):
    """Checks if the label is contained in the model's answer (case-insensitive)."""
    if model_ans is None: return "❌ ERROR"
    model_ans_clean = model_ans.lower().strip()
    
    if isinstance(label, str):
        valid_labels = [label]
    else:
        valid_labels = label 
        
    for valid_label in valid_labels:
        # Relaxed check: is the ground truth inside the answer?
        if str(valid_label).lower() in model_ans_clean:
            return "✅ CORRECT"
            
    return "❌ INCORRECT"

# 2. Execution Logic
train_df = load_chart_qa_split("train", type="human")

if train_df is not None:
    # Run the first 2 tasks as a test
    tasks_to_run = train_df.head(2)
    
    for index, entry in tasks_to_run.iterrows():
        print("\n" + "="*80)
        print(f"🚀 PROCESSING CHART TASK {index + 1} | Query: {entry['query']}")
        print("="*80)

        # 1. Open the image ONCE per task (for human review)
        try:
            print(f"Opening image: {Path(entry['image_path']).name}")
            img = Image.open(entry['image_path'])
            img.show()
        except Exception as e:
            print(f"Warning: Could not open image: {e}")

        # 2. Loop through ALL 3 models for this single task
        for model_name in MODELS_TO_TEST:
            print(f"\nTesting Model: {model_name}...")
            
            try:
                # Call API
                model_reply, run_time = get_chart_response(model_name, entry['query'], entry['image_path'])
                
                # Check Answer
                correctness = check_correctness(model_reply, entry['label'])
                
                # Display Results
                print("-" * 60)
                print(f"📊 RESULT: {model_name}")
                print(f"⏱️  Time:         {run_time:.2f}s")
                print(f"📝 Prompt:       {entry['query']}")
                print(f"🎯 Ground Truth: {entry['label']}")
                print(f"🧐 Reply:        {model_reply}")
                print(f"🏁 Verdict:      {correctness}")
                print("-" * 60)

            except Exception as e:
                print(f"❌ ERROR with {model_name}: {e}")

            # 3. Wait 5 seconds between EACH run (to let GPU cool/swap)
            print("⏳ Cooling down (5s)...")
            time.sleep(5)


🚀 PROCESSING CHART TASK 1 | Query: Is the value of Favorable 38 in 2015?
Opening image: 10095.png

Testing Model: llava:7b...
------------------------------------------------------------
📊 RESULT: llava:7b
⏱️  Time:         6.22s
📝 Prompt:       Is the value of Favorable 38 in 2015?
🎯 Ground Truth: Yes
🧐 Reply:         Yes, the value of Favorable is 38 in 2015. 
🏁 Verdict:      ✅ CORRECT
------------------------------------------------------------
⏳ Cooling down (5s)...

Testing Model: qwen2.5vl:7b...
------------------------------------------------------------
📊 RESULT: qwen2.5vl:7b
⏱️  Time:         54.01s
📝 Prompt:       Is the value of Favorable 38 in 2015?
🎯 Ground Truth: Yes
🧐 Reply:        Yes
🏁 Verdict:      ✅ CORRECT
------------------------------------------------------------
⏳ Cooling down (5s)...

Testing Model: qwen3-vl:8b...
------------------------------------------------------------
📊 RESULT: qwen3-vl:8b
⏱️  Time:         10.32s
📝 Prompt:       Is the value of Favorabl